In [15]:
import sys, os
# add ../.. to path
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import control as ct
import numpy as np

import models.model_coef as TF
from models.mav_dynamics_control import MavDynamics
from models.compute_models import euler_state, quaternion_state
import parameters.control_parameters as AP
import parameters.aerosonde_parameters as MAV
import parameters.simulation_parameters as SIM
from message_types.msg_delta import MsgDelta

np.set_printoptions(suppress=True, precision=4, linewidth=120)


mav = MavDynamics(SIM.ts_simulation)
m = MAV.mass
g = MAV.gravity
u_star = AP.u_star
w_star = AP.w_star

x_trim = euler_state(TF.x_trim)
u_trim = TF.u_trim

def x_dot_euler(xdot_quat, x_euler, x_quat):
    dTe_dxq_of_xq = np.zeros((x_euler.size, x_quat.size))
    Te_out = x_euler
    n = x_quat.size
    eps = 0.001
    for i in range(n):
        x_quat_i = x_quat + eps * np.eye(n)[:, [i]]
        Te_outi = euler_state(x_quat_i)
        dTe_dxq_of_xq[:, [i]] = (Te_outi - Te_out) / eps
    xdot_euler = dTe_dxq_of_xq @ xdot_quat
    return xdot_euler

def uparms_update(t, x, u, params):
    x_euler = x.reshape((-1,1))
    x_quat = quaternion_state(x_euler)
    u = u.reshape((-1,1))
    delta = MsgDelta()
    delta.from_array(np.pad(u, ((0,2),(0,0)), constant_values=0))
    wind = np.zeros((6, 1))

    mav._state = x_quat  # x_quat is already the quaternion state
    mav._update_velocity_data(wind)
    forces_moments = mav._forces_moments(delta)
    xdot_quat = mav._f(x_quat, forces_moments)

    xdot_euler = x_dot_euler(xdot_quat, x_euler, x_quat)
    return xdot_euler.flatten()


def uparms_output(t, x, u, params):
    # Derive accelerations from dynamics so the output is a true function of (x, u),
    # not a time-history finite difference — required for correct ct.linearize Jacobians.
    xdot = uparms_update(t, x, u, params)
    h = x[2]*-1
    u_body = x[3]
    v_body = x[4]
    w_body = x[5]
    udot   = xdot[3]
    vdot   = xdot[4]
    wdot   = xdot[5]
    theta  = x[7]
    V      = np.sqrt(u_body**2 + v_body**2 + w_body**2)
    Vgdot  = (u_body*udot + v_body*vdot + w_body*wdot) / V
    Edot   = Vgdot / MAV.gravity + np.sin(theta)
    Ldot   = Vgdot / MAV.gravity - np.sin(theta)
    E = 0.5*m*V**2 + m*g*h
    L = 0.5*m*V**2 - m*g*h
    return np.array([E, L])


# Instantiate system
sys_nl = ct.NonlinearIOSystem(uparms_update, uparms_output, states=12, inputs=4, outputs=2)

# Linearize at trim
x_eq = x_trim.flatten()
u_eq = u_trim.flatten()
sys_lin = ct.linearize(sys_nl, x_eq, u_eq)

A = sys_lin.A
B = sys_lin.B
C = sys_lin.C
D = sys_lin.D

E1 = np.array([
    [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
    [0, 0, -1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
])
E2 = np.array([
    [1, 0, 0, 0],
    [0, 0, 0, 1]
])
E3 = np.array([
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1],
    [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0,  0, 0, 0, 1, 0, 0, 0]
])
E4 = np.array([
    [0, 1, 0, 0],
    [0, 0, 1, 0]
])

A_lon = E1 @ A @ E1.T
B_lon = E1 @ B @ E2.T
C_lon = C @ E1.T
D_lon = D @ E2.T

A_lat = E3 @ A @ E3.T
B_lat = E3 @ B @ E4.T
C_lat = C @ E3.T
D_lat = D @ E4.T

print(f"A_lon =\n{A_lon}")
print(f"B_lon =\n{B_lon}")
print(f"C_lon =\n{C_lon}")
print(f"D_lon =\n{D_lon}")

A_lon =
[[-0.2822  0.4946 -1.2123 -9.7979  0.    ]
 [-0.5611 -4.4978 24.3714 -0.4874  0.    ]
 [ 0.1986 -3.993  -5.2947  0.      0.    ]
 [ 0.      0.      1.0001  0.      0.    ]
 [ 0.0497 -0.9988  0.     25.      0.    ]]
B_lon =
[[ -0.1392   9.4813]
 [ -2.5861   0.    ]
 [-36.1124   0.    ]
 [  0.       0.    ]
 [  0.       0.    ]]
C_lon =
[[ 274.6604   13.662     0.        0.      107.91  ]
 [ 274.6604   13.662     0.        0.     -107.91  ]]
D_lon =
[[0. 0.]
 [0. 0.]]


In [16]:
import numpy as np
from numpy import concatenate, zeros, eye, diag
np.set_printoptions(suppress=True, precision=4, linewidth=120)

# Trim values for linearized Va output
u_star_lon = TF.x_trim.item(3)
w_star_lon = TF.x_trim.item(5)
Va_star    = TF.Va_trim

# Augmented state: [u, w, q, theta, h | Edot_int, Ldot_int | h_int, Va_int]
n_lon   = A_lon.shape[0]   # 5
n_tecs  = C_lon.shape[0]   # 2
n_aug   = n_lon + n_tecs   # 7

Ahat = concatenate([
    concatenate([A_lon,   zeros((n_lon,   n_tecs))], axis=1),
    concatenate([C_lon,   zeros((n_tecs,  n_tecs))], axis=1),
], axis=0)

Bhat = concatenate([B_lon, D_lon], axis=0)

# print("Ahat ="); print(Ahat)
# print("\nBhat ="); print(Bhat)
print(f"Ahat {Ahat.shape}")
print(f"Bhat {Bhat.shape}")

Ahat (7, 7)
Bhat (7, 2)


In [17]:
from numpy import diag
from scipy.linalg import solve_continuous_are, inv

#                  u      w      q      theta  h      E     L
Qlon = diag([    10.0,  100.0,  100.0,  10.0,  50.0,  60.0,  50.0])
Rlon = diag([1.0, 1.0])   # elevator, throttle

Plon = solve_continuous_are(Ahat, Bhat, Qlon, Rlon)
Klon = inv(Rlon) @ Bhat.T @ Plon

labels = ['u', 'w', 'q', 'theta', 'h', 'Edot_int', 'Ldot_int', 'h_int', 'Va_int']
print(f"Klon (de | dt) x [{', '.join(labels)}]")
print(Klon)

Klon (de | dt) x [u, w, q, theta, h, Edot_int, Ldot_int, h_int, Va_int]
[[   -0.1702    34.657    -13.3336 -1243.5042  -334.5557    -5.2196     5.2246]
 [   24.8184     1.4795    -0.1569   -10.7555    -0.6616     5.7233     4.7648]]


In [18]:
"""
Pole placement strategy using
x = [u, w, q, theta, h, Edot_int, Ldot_int]
"""
import control as ct
import numpy as np
np.set_printoptions(suppress=True, precision=2, linewidth=120)


App = Ahat[0:7, 0:7]
Bpp = Bhat[0:7,:]
labels = ['u', 'w', 'q', 'theta', 'h', 'E', 'L']

controllability_matrix = ct.ctrb(App, Bpp)
c_rank = np.linalg.matrix_rank(controllability_matrix)

eigenvalues, eigenvectors = np.linalg.eig(App)

# print(f"App = \n{App}\nBpp = \n{Bpp}")
print(f"rank = {c_rank}")
print(f"\n{'Eigenvalue':<30} {'Eigenvector (column)'}")
print("-" * 80)
for i, (val, vec) in enumerate(zip(eigenvalues, eigenvectors.T)):
    vec_str = np.array2string(vec, formatter={'complexfloat': lambda x: f'{x.real:+.2f}{x.imag:+.2f}j' if x.imag != 0 else f'{x.real:+.2f}'})
    print(f"λ{i} = {val:+.2f}   {vec_str}")

rank = 7

Eigenvalue                     Eigenvector (column)
--------------------------------------------------------------------------------
λ0 = +0.00+0.00j   [+0.00 +0.00 +0.00 +0.00 +0.00 +1.00 +0.00]
λ1 = +0.00+0.00j   [+0.00 +0.00 +0.00 +0.00 +0.00 +0.00 +1.00]
λ2 = +0.00+0.00j   [+0.00 +0.00 +0.00 +0.00 +0.00 -0.71 +0.71]
λ3 = -0.14+0.48j   [-0.00+0.00j -0.00+0.00j -0.00+0.00j +0.00+0.00j -0.00-0.00j -0.00+0.29j +0.96]
λ4 = -0.14-0.48j   [-0.00-0.00j -0.00-0.00j -0.00-0.00j +0.00-0.00j -0.00+0.00j -0.00-0.29j +0.96]
λ5 = -4.89+9.87j   [-0.03-0.00j +0.77 -0.01+0.31j +0.03-0.01j -0.02+0.02j +0.12-0.16j -0.43-0.29j]
λ6 = -4.89-9.87j   [-0.03+0.00j +0.77 -0.01-0.31j +0.03+0.01j -0.02-0.02j +0.12+0.16j -0.43+0.29j]


In [ ]:
import control as ct
import numpy as np

# 2. Define the desired closed-loop pole locations
# We want to place them further into the left-half plane for speed and stability
desired_poles = [-10.1, -9, -11, -10.89+9.87j, -10.89-9.87j, -10+0.48j, -10-0.48j]

# 3. Compute the feedback gain matrix K
# u = -Kx, so the closed-loop system matrix becomes (A - B*K)
K = ct.place(App, Bpp, desired_poles)

print("Calculated Feedback Gain Matrix (K):")
print(np.array2string(K, separator=', '))

# 4. Verification: Create the closed-loop system and find its poles
A_closed = App - Bpp @ K
actual_poles = np.linalg.eigvals(A_closed)

print("\nVerification of Closed-Loop Poles:")
print(actual_poles)

Calculated Feedback Gain Matrix (K):
[[ -0.06,   2.32,  -1.36, -81.02, -17.72,  -0.18,   0.18],
 [  1.97,   2.06,  -0.55, -64.58, -18.3 ,  -0.22,   0.26]]

Verification of Closed-Loop Poles:
[-10.89+3.87j -10.89-3.87j  -9.  +0.j   -10.  +0.48j -10.  -0.48j -10.1 +0.j   -11.  +0.j  ]


/Users/marciano/Repos/mavsim/venv/lib/python3.10/site-packages/control/statefbk.py:102: UserWarning: Convergence was not reached after maxiter iterations.
You asked for a tolerance of 0.001, we got 0.9999914109931194.
  result = place_poles(A_mat, B_mat, placed_eigs, method='YT')
